In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

h5_path = "/home/robot/robot_ws/egomimic/robot/demos/demo_	1.hdf5"  # <- change this

with h5py.File(h5_path, "r") as f:
    # 1) load action
    cmd_actions = f["actions"]["eepose"][...]  # shape: (T, 14) based on your saver
    robot_actions = f["observations"]["eepose"][...]
    T = cmd_actions.shape[0]
    print("cmd_actions shape:", cmd_actions.shape)
    print("robot_actions shape:", robot_actions.shape)

    # left arm: 0:3 xyz, 3:6 ypr, 13 gripper
    # right arm: 7:10 xyz, 10:13 ypr, 14 gripper

    arms = ["left", "right"]
    dims = ["x", "y", "z", "roll", "pitch", "yaw", "gripper"]
    # index map
    sel = {
        "left": {
            "xyz": slice(0, 3),
            "ypr": slice(3, 6),
            "gripper": 6,
        },
        "right": {
            "xyz": slice(7, 10),
            "ypr": slice(10, 13),
            "gripper": 13,
        },
    }

    # ---- plot for each arm, each variable ---
    for arm in arms:
        fig, axs = plt.subplots(7, 1, figsize=(12, 14), sharex=True)
        fig.suptitle(
            f"{arm.capitalize()} Arm: Commanded vs Robot (xyz, roll, pitch, yaw, gripper)"
        )

        # xyz
        for i, label in enumerate(["x", "y", "z"]):
            axs[i].plot(
                np.arange(T), cmd_actions[:, sel[arm]["xyz"].start + i], label="cmd"
            )
            axs[i].plot(
                np.arange(T), robot_actions[:, sel[arm]["xyz"].start + i], label="robot"
            )
            axs[i].set_ylabel(label)
            axs[i].legend(loc="upper right")
            axs[i].set_title(f"{label} position")

        # roll, pitch, yaw (note: these are YPR, so roll=y, pitch=p, yaw=r)
        for j, label in enumerate(["roll", "pitch", "yaw"]):
            axs[3 + j].plot(
                np.arange(T), cmd_actions[:, sel[arm]["ypr"].start + j], label="cmd"
            )
            axs[3 + j].plot(
                np.arange(T), robot_actions[:, sel[arm]["ypr"].start + j], label="robot"
            )
            axs[3 + j].set_ylabel(label)
            axs[3 + j].legend(loc="upper right")
            axs[3 + j].set_title(f"{label} (rad)")

        # gripper
        axs[6].plot(np.arange(T), cmd_actions[:, sel[arm]["gripper"]], label="cmd")
        axs[6].plot(np.arange(T), robot_actions[:, sel[arm]["gripper"]], label="robot")
        axs[6].set_ylabel("gripper")
        axs[6].legend(loc="upper right")
        axs[6].set_title("Gripper")

        axs[-1].set_xlabel("timestep")
        plt.tight_layout()
        plt.show()

    # ---- show images ----
    cam_group = f["observations"]["images"]
    cam_names = list(cam_group.keys())
    print("cams:", cam_names)

    # 3) sample a few timesteps to show images
    timesteps_to_show = [0, min(10, T - 1), T - 1]  # first, 10th, last

    for cam in cam_names:
        imgs = cam_group[cam]  # (T, 480, 640, 3)
        print(f"{cam} shape:", imgs.shape)

        fig, axs = plt.subplots(
            1, len(timesteps_to_show), figsize=(4 * len(timesteps_to_show), 4)
        )
        if len(timesteps_to_show) == 1:
            axs = [axs]
        for j, t in enumerate(timesteps_to_show):
            axs[j].imshow(imgs[t])
            axs[j].set_title(f"{cam} @ t={t}")
            axs[j].axis("off")
        plt.show()

In [ ]:
import h5py
import os
from PIL import Image  # pillow

h5_path = "/home/robot/robot_ws/egomimic/robot/demos/demo_2.hdf5"  # change
out_dir = "./frames"
os.makedirs(out_dir, exist_ok=True)

with h5py.File(h5_path, "r") as f:
    img_grp = f["observations"]["images"]
    cam_names = list(img_grp.keys())
    print("cams:", cam_names)

    for cam in cam_names:
        frames = img_grp[cam]  # (T, H, W, 3), uint8, RGB
        T = frames.shape[0]
        for i in range(T):
            rgb = frames[i]  # numpy array
            img = Image.fromarray(rgb)  # RGB already
            fname = os.path.join(out_dir, f"{cam}_{i:05d}.png")
            img.save(fname)

print(f"saved frames to {out_dir}")